In [1]:
import torch
import torch.nn as nn

# Последовательная модель

## Первый способ создания

In [2]:
"""
    input_size => (batch_size, C, H, W)
"""

model = nn.Sequential(
    nn.Conv2d(3, 32, (3, 3)),   # inp => (batch_size, 3, 28, 28), out => (batch_size, 32, 26, 26)
    nn.ReLU(),
    nn.Conv2d(32, 64, (3, 3))   # inp => (batch_size, 32, 26, 26), out => (batch_size, 64, 24, 24)
)

model

Sequential(
  (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1))
  (1): ReLU()
  (2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
)

## Второй способ создания

In [3]:
model = nn.Sequential()
model.add_module('layer_1', nn.Conv2d(3, 32, (3, 3)))   # inp => (batch_size, 3, 28, 28), out => (batch_size, 32, 26, 26)
model.add_module('relu', nn.ReLU())
model.add_module('layer_2', nn.Conv2d(32, 64, (3, 3)))  # inp => (batch_size, 32, 26, 26), out => (batch_size, 64, 24, 24)

In [4]:
model

Sequential(
  (layer_1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1))
  (relu): ReLU()
  (layer_2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
)

In [5]:
input = torch.rand([16, 3, 28, 28], dtype=torch.float32)

out = model(input)
out.shape

torch.Size([16, 64, 24, 24])

In [7]:
# model.state_dict()
model.state_dict()['layer_1.weight'].shape
# model.state_dict()['layer_1.bias'].shape
# model.state_dict()['layer_2.weight'].shape

torch.Size([32, 3, 3, 3])

# Создание класса для модели нейронной сети

In [ ]:
class MyModel(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv_1 = nn.Conv2d(in_channels, 32, (3, 3))   # inp => (batch_size, 3, 50, 50), out => (batch_size, 32, 48, 48)
        self.conv_2 = nn.Conv2d(32, out_channels, (3, 3))  # inp => (batch_size, 32, 48, 48), out => (batch_size, out_channels, 46, 46)
        self.act = nn.ReLU()

    def forward(self, x):
        x = self.conv_1(x)
        x = self.act(x)
        out = self.conv_2(x)
        return out

model = MyModel(3, 64)
model

## Выравнивание для подачи в линейный слой

### Reshape

In [8]:
tensor = torch.rand([2, 1, 3, 3], dtype=torch.float32)
tensor

tensor([[[[0.9009, 0.9971, 0.5188],
          [0.5626, 0.9367, 0.2987],
          [0.3939, 0.1064, 0.2019]]],


        [[[0.4086, 0.4019, 0.2442],
          [0.5481, 0.8418, 0.1299],
          [0.9845, 0.3002, 0.0616]]]])

In [9]:
conv = nn.Conv2d(1, 3, (3, 3))(tensor)
print(conv.shape)
print(conv)

torch.Size([2, 3, 1, 1])
tensor([[[[0.2384]],

         [[0.3574]],

         [[0.0702]]],


        [[[0.3048]],

         [[0.2271]],

         [[0.1540]]]], grad_fn=<ConvolutionBackward0>)


In [10]:
tensor = tensor.reshape([2, -1])
print(tensor.shape)
print(tensor)

torch.Size([2, 9])
tensor([[0.9009, 0.9971, 0.5188, 0.5626, 0.9367, 0.2987, 0.3939, 0.1064, 0.2019],
        [0.4086, 0.4019, 0.2442, 0.5481, 0.8418, 0.1299, 0.9845, 0.3002, 0.0616]])


In [11]:
linear = nn.Linear(9,10)(tensor)
print(linear.shape)
print(linear)

torch.Size([2, 10])
tensor([[-0.3760,  0.4970, -0.5575, -0.4067,  0.4297,  0.3993, -0.1372,  0.7000,
          0.1664, -0.5472],
        [-0.0642,  0.3738, -0.1966, -0.6026,  0.3785,  0.1343,  0.0104,  0.3544,
          0.0566, -0.3001]], grad_fn=<AddmmBackward0>)


### Flatten

In [12]:
tensor = torch.rand([2, 1, 3, 3], dtype=torch.float32)
tensor

tensor([[[[0.2167, 0.3324, 0.4348],
          [0.0741, 0.7726, 0.8197],
          [0.1729, 0.1594, 0.8845]]],


        [[[0.1359, 0.2282, 0.0917],
          [0.2727, 0.1858, 0.6719],
          [0.5181, 0.0171, 0.8194]]]])

In [14]:
# flat = tensor.flatten(start_dim=1, end_dim=-1)
flat = nn.Flatten()(tensor)

print(flat.shape)
print(flat)

torch.Size([2, 9])
tensor([[0.2167, 0.3324, 0.4348, 0.0741, 0.7726, 0.8197, 0.1729, 0.1594, 0.8845],
        [0.1359, 0.2282, 0.0917, 0.2727, 0.1858, 0.6719, 0.5181, 0.0171, 0.8194]])


# Создание свёрточной нейронной сети

Все нейронные сети собираются под конкретные входные данные, поэтому *input* отсутствует

In [15]:
class MyModel(nn.Module):
    def __init__(self, out):
        super().__init__()
        self.conv_1 = nn.Conv2d(3, 32, (3, 3))    # (batch_size, 3, 28, 28) => (batch_size, 32, 26, 26)
        self.conv_2 = nn.Conv2d(32, 64, (3, 3))   # (batch_size, 32, 26, 26) => (batch_size, 64, 24, 24)

        self.flatten = nn.Flatten()    # (batch_size, 64, 24, 24) => (batch_size, 64*24*24)

        self.act = nn.ReLU()

        self.layer_1 = nn.Linear( 64*24*24 , 128)
        self.layer_2 = nn.Linear(128, out)

    def forward(self, x):
        x = self.conv_1(x)
        x = self.act(x)
        x = self.conv_2(x)

        x = self.flatten(x)

        x = self.layer_1(x)
        x = self.act(x)
        out = self.layer_2(x)
        return out

model = MyModel(10)
model

MyModel(
  (conv_1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1))
  (conv_2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (act): ReLU()
  (layer_1): Linear(in_features=36864, out_features=128, bias=True)
  (layer_2): Linear(in_features=128, out_features=10, bias=True)
)

In [16]:
input = torch.rand([16, 3, 28, 28], dtype=torch.float32)

out = model(input)
out.shape

torch.Size([16, 10])

Более сложная модель

In [17]:
class MyModel(nn.Module):
    def __init__(self, in_channels, out):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 32, (3, 3), bias=False),  # (batch_size, 3, 28, 28) => (batch_size, 32, 26, 26)
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, (3, 3), bias=False),   # (batch_size, 32, 26, 26) => (batch_size, 64, 24, 24)
            nn.BatchNorm2d(64),
            nn.ReLU()
        )

        self.flatten = nn.Flatten()    # (batch_size, 64, 24, 24) => (batch_size, 64*24*24)

        self.fc = nn.Sequential(
            nn.Linear(64*24*24, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, out)
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.flatten(x)
        out = self.fc(x)
        return out

model = MyModel(3, 10)
model

MyModel(
  (conv): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), bias=False)
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), bias=False)
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
  )
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc): Sequential(
    (0): Linear(in_features=36864, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.5, inplace=False)
    (3): Linear(in_features=128, out_features=10, bias=True)
  )
)

In [18]:
input = torch.rand([16, 3, 28, 28], dtype=torch.float32)

out = model(input)
out.shape

torch.Size([16, 10])

## Создание модели для классификации MNIST

In [19]:
class ModelMNIST(nn.Module):
    def __init__(self, in_channels, out):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 32, (3, 3), bias=False),   # (batch_size, 1, 28, 28) => (batch_size, 32, 26, 26)
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, (3, 3), bias=False),    # (batch_size, 32, 26, 26) => (batch_size, 64, 24, 24)
            nn.BatchNorm2d(64),
            nn.ReLU()
        )

        self.flatten = nn.Flatten()    # (batch_size, 64, 24, 24) => (batch_size, 64*24*24)

        self.fc = nn.Sequential(
            nn.Linear(64*24*24, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, out)
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.flatten(x)
        out = self.fc(x)
        return out

In [20]:
model_classification = ModelMNIST(1, 10)

In [21]:
# Определяем функцию потерь и оптимизатор градиентного спуска.
loss_classification = nn.CrossEntropyLoss()
opt_classification = torch.optim.Adam(model_classification.parameters(), lr=0.001)

In [22]:
# Проверяем правильность построения модели.
input = torch.rand([16, 1, 28, 28], dtype=torch.float32)

out = model_classification(input)
out.shape  # (16, 10)

torch.Size([16, 10])

## Модель для задачи регрессии

In [23]:
class MyModel(nn.Module):
    def __init__(self, in_channels, out):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 32, (3, 3), bias=False),  # (batch_size, 1, 64, 64) => (batch_size, 32, 62, 62)
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, (3, 3), bias=False),   # (batch_size, 32, 62, 62) => (batch_size, 64, 60, 60)
            nn.BatchNorm2d(64),
            nn.ReLU()
        )

        self.flatten = nn.Flatten()    # (batch_size, 64, 60, 60) => (batch_size, 64*60*60)

        self.fc = nn.Sequential(
            nn.Linear(64*60*60, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, out)
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.flatten(x)
        out = self.fc(x)
        return out

In [24]:
model_regression = MyModel(1, 2)

In [25]:
loss_regression = nn.MSELoss()
opt_regression = torch.optim.Adam(model_regression.parameters(), lr=0.001)

In [26]:
input = torch.rand([16, 1, 64, 64], dtype=torch.float32)

out = model_regression(input)
out.shape   # (16, 2)

torch.Size([16, 2])